In [137]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [138]:
#FORWARD KINEMATICS - lEG

#GLOBAL DEFS
#1.28:1 max thigh to shin ratio (1.25 used)
L1 = 2.0 #Thigh
L2 = 1.6 #Shin
x0 = 2.0 #X offset
y0 = 4.0 #Y offset- init HIP height
theta1 = -np.pi / 4 #HIP Angle. Neg given CW rot.
theta2 = -np.pi / 12 #KNEE Angle. Neg, relative to thigh. CW.

def forward_kinematics(L1, L2, theta1, theta2, x0 = 0.0, y0 = 0.0):
    #Initial offset. From base (x0, y0) to (x1, y1)
    x1 = L1 * np.cos(theta1)
    y1 = L1 * np.sin(theta1)

    #Second Offset. From (x1, y1) to (x2, y2) Final.
    #theta1 + theta2 full angle rot at x2, y2 tith respect to base horizontal. CCW.
    x2 = x1 + L2 * np.cos(theta1 + theta2)
    y2 = y1 + L2 * np.sin(theta1 + theta2)

    #Base point, First Offset, Second offset. Real res: x0 + x2
    xf = np.array([x0, x0 + x1, x0 + x2])
    yf = np.array([y0, y0 + y1, y0 + y2])
    
    return (xf, yf)

#Motion setup
fk_frames = 100
t = 2 * np.linspace(0, np.pi, fk_frames)

hip_amp = np.pi / 8
knee_amp = np.pi / 6

#angle sequence vals for ploting
hip_ang_seq = theta1 + hip_amp * np.sin(t)
knee_ang_seq = theta2 - knee_amp * abs(np.sin(t))

#Plot setup

fig, ax = plt.subplots(figsize = (8, 8))
ax.set_xlim(x0 - L1 - L2, x0 + L1 + L2)
ax.set_ylim(y0 - L1 - L2, y0 + 1)
ax.grid(True)
ax.set_title("Leg Waking Cycle - Forward Kinematics")

x_init, y_init = forward_kinematics(L1, L2, hip_ang_seq[0], knee_ang_seq[0], x0, y0)
links, = ax.plot(x_init, y_init, 'g-', linewidth = 6, label = 'Leg Links')
joints, = ax.plot(x_init, y_init, 'ko', markersize = 10)

def init_fk_elements():
    links.set_data([], [])
    joints.set_data([], [])
    
    return (links, joints)



#ANIMATION
def update_fk_leg(frame):
    haf_seq = hip_ang_seq[frame]
    kaf_seq = knee_ang_seq[frame]

    x_i, y_i = forward_kinematics(L1, L2, haf_seq, kaf_seq, x0, y0)

    links.set_data(x_i, y_i)
    joints.set_data(x_i, y_i)
    
    return (links, joints)

FPS = 30
fk_leg_ani = FuncAnimation( 
                             fig, 
                             update_fk_leg, 
                             frames = fk_frames, 
                             interval = 50,       
                             blit = True,
                             init_func = init_fk_elements,
                             repeat = True        
                          )
plt.close()
fk_leg_ani.save("../res/FK_walking_cycle.gif", writer = "pillow", fps = FPS)

In [139]:
#INVERSE KINEMATICS - CRANE

#GLOBAL DEFS
CL1 = 7.5
CL2 = 2.5
L1 = CL1
L2 = CL2

def inverse_kinematics(L1, L2, x_f, y_f):
    r = np.sqrt(x_f ** 2 + y_f ** 2)

    #Edge Case. Impossible. if r > L1 + L2 or r < abs(L1 - L2)
    #You fucked up.
    if r < 0 or r > L1 + L2 or r < abs(L2 - L1):
        return (np.nan, np.nan)
        
    #THETA2
    #L1, L2, r triangle. CW Angle i.e. neg angle
    cos_gamma = (L1 ** 2 + L2 ** 2 - r ** 2) / (2 * L1 * L2)
    cos_gamma_clip = np.clip(cos_gamma, -1.0, 1.0)
    gamma = np.arccos(cos_gamma_clip)
    theta2 = -(np.pi - gamma)

    #THETA 1 = alpha - theta2
    alpha = np.atan2(y_f, x_f)
    cos_beta = (L1 ** 2 + r ** 2 - L2 ** 2) / (2 * L1 * r)
    cos_beta_clip = np.clip(cos_beta, -1.0, 1.0)
    beta = np.arccos(cos_beta_clip)
    theta1 = alpha - beta

    return (theta1, theta2)

#Motion setup
P1 = np.array([2.0, 6.5])  
P2 = np.array([4.0, 4.0])  
P3 = np.array([7.0, 2.5])  
P4 = np.array([9.0, 1.5])  

step = 50

#LERP LARP
# Close gap
t1 = np.linspace(0, 1, step, endpoint=False) 
seg1_x = P1[0] + (P2[0] - P1[0]) * t1
seg1_y = P1[1] + (P2[1] - P1[1]) * t1

# Move Horizontal
t2 = np.linspace(0, 1, step, endpoint=False) 
seg2_x = P2[0] + (P3[0] - P2[0]) * t2
seg2_y = P2[1] + (P3[1] - P2[1]) * t2

# Pick up up (P3 to P4)
t3 = np.linspace(0, 1, step, endpoint=False)
seg3_x = P3[0] + (P4[0] - P3[0]) * t3
seg3_y = P3[1] + (P4[1] - P3[1]) * t3

# Return (P4 to P1)
t4 = np.linspace(0, 1, step)
seg4_x = P4[0] + (P1[0] - P4[0]) * t4
seg4_y = P4[1] + (P1[1] - P4[1]) * t4

target_x_seq = np.concatenate([seg1_x, seg2_x, seg3_x, seg4_x])
target_y_seq = np.concatenate([seg1_y, seg2_y, seg3_y, seg4_y])

crane_targets = list(zip(target_x_seq, target_y_seq))

ik_frames = len(crane_targets)

#Plot setup. Center at origin.
fig, ax = plt.subplots(figsize = (8, 8))
ax.set_xlim(- CL1 - CL2, CL1 + CL2)
ax.set_ylim(-5, CL1 + CL2)
ax.grid(True)
ax.set_title("Crane Pick Up Cycle - Inverse Kinematics")

crane_arm, = ax.plot([], [], 'o-', lw = 5, color = 'blue', label = 'Crane Arm')
crane_hk, = ax.plot([], [], 'o', ms = 10, color = 'red', label = 'Hook Target')

# Initialization function for the animation
def init_ik_elements():
    crane_arm.set_data([], [])
    crane_hk.set_data([], [])
    
    return (crane_arm, crane_hk)


def update_ik_crane(frame):
    
    x_f, y_f = crane_targets[frame]

    theta1, theta2 = inverse_kinematics(L1, L2, x_f, y_f)

    #Previous FK impl. Execute previous cell or drag the func code to this one.
    x_i, y_i = forward_kinematics(L1, L2, theta1, theta2)
    crane_arm.set_data(x_i, y_i)
    crane_hk.set_data([x_i[-1]], [y_i[-1]])
    
    return (crane_arm, crane_hk)

FPS = 30
ik_crane_ani = FuncAnimation( 
                             fig, 
                             update_ik_crane, 
                             frames = ik_frames, 
                             interval = 50,  
                             init_func = init_ik_elements,
                             blit = True,         
                             repeat = True        
                          )
plt.close()
ik_crane_ani.save("../res/IK_Crane_Lifting.gif", writer = "pillow", fps = FPS)